# 08 — XAI ve Güvenilirlik

**Kapsam:** Güvenilirlik, açıklanabilirlik (XAI) ve sorumlu yapay zekâ uygulamaları —
halüsinasyon azaltma, önyargı/sır sızıntısı tespiti.

Bu notebook üç aracı gösterir:
1. Halüsinasyon tespiti (NLI tabanlı groundedness skoru) — üretilen dokümanın, verilen
   şablon/taslak notlarda olmayan bir özellik/parametre "uydurup uydurmadığını" ölçer
2. Attribution — üretilen dokümanın hangi kaynak şablondan/cümleden geldiğini gösterme
3. Güvenlik kontrolü — offensive dil + örnek kod bloklarına sızabilecek gerçek görünümlü
   kimlik bilgisi (API anahtarı, şifre) taraması

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) hiçbir şey yapmadan devam eder.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
import os, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"

if not os.path.exists(PROJECT_DIR):
    try:
        from google.colab import drive
        # drive.mount() zaten mount edilmişse anında geri döner (idempotent);
        # os.path.exists("/content/drive") ile "mount edilmiş mi" kontrol etmek
        # güvenilmez çünkü klasör, başarısız/yarım bir mount denemesinden sonra
        # bile var olabilir. Bu yüzden koşulsuz çağırıyoruz.
        drive.mount("/content/drive", force_remount=True)
        if os.path.exists(DRIVE_ZIP_PATH):
            import shutil
            shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
        else:
            print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
                  "yükleyin ya da kendi reponuzu klonlayın: "
                  f"!git clone <repo-url> {PROJECT_DIR}")
    except ImportError:
        pass  # Colab dışında (yerelde) çalışıyorsanız bu adım gerekmez.

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)


## 1. Halüsinasyon Tespiti

In [ ]:
from src.rag.rag_pipeline import answer
from src.xai.hallucination_check import groundedness_score, flag_unsupported_sentences

result = answer("İyi bir API dokümantasyonu hangi bölümleri içermeli?")
contexts = [c["text"] for c in result["contexts"]]

score = groundedness_score(result["answer"], contexts)
print(f"Groundedness skoru: {score:.2f} (1.0 = tamamen kaynağa/şablona dayalı)")

flagged = flag_unsupported_sentences(result["answer"], contexts)
for f in flagged:
    print("Desteksiz cümle:", f["sentence"], f"(skor: {f['entailment_score']:.2f})")


## 2. Attribution

Cevabın hangi chunk'lardan/cümlelerden geldiğini gösterelim.

In [ ]:
from src.xai.attribution import chunk_level_attribution, sentence_level_attribution

chunk_scores = chunk_level_attribution(result["answer"], result["contexts"])
for c in chunk_scores:
    print(f"{c['contribution_score']:.3f} — {c['doc_title']}")

print()
sentence_matches = sentence_level_attribution(result["answer"], result["contexts"])
for m in sentence_matches[:3]:
    print("Cevap cümlesi:", m["answer_sentence"])
    print("  -> En yakın kaynak:", m["best_match"], f"(benzerlik: {m['similarity']:.2f})")


## 3. Güvenlik Kontrolü (offensive dil + sır sızıntısı)

In [ ]:
from src.xai.safety_check import safety_check

check = safety_check(result["answer"])
print("Güvenli mi:", check["is_safe"])
print("Model kontrolü:", check["model_check"])
print("Olası sır sızıntısı eşleşmeleri:", check["secret_hits"])


## Sonuç

Bu 8 notebook'u tamamladıysanız, RAG mimarisinden LLM fine-tuning'e, alignment'tan
LLMOps ve XAI'ye kadar uzanan tüm süreci uçtan uca, çalışan bir sistem üzerinde
deneyimlemiş oldunuz. Sonraki adım: `docker/` ile servisi konteynerleştirip
`.github/workflows/ci.yml` ile CI'ye bağlamak (bkz. proje README'si).